In [20]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
import re

# root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
# filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'
filename = "/home/lwu/ARCHS4/Tests/top_10_genesets_per_category/test_cases/test_h5df_data.hdf5"
with h5py.File(filename, 'r') as f:
    #low_cat_matrix = f['data/lowest_tier_category_matrix2'][:]
    #high_cat_matrix = f['data/highest_tier_category_matrix2'][:] # Done later
    arch_gene_names = f['meta/genes/all_gene_names'][:]
    category_names = f['meta/biosamples/category'][:]

arch_category = [x.decode('UTF-8') for x in category_names]
arch_gene_names = [x.decode('UTF-8') for x in arch_gene_names]

category_list  = ["Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle","HUVEC", 
    "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
    "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF",
    "HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
    "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
    "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
    "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus","Granulocyte", "Neutrophil", "Blymphocyte", 
    "Plasma Cell", "Plasmacytoid Dendritic Cell", 
    "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
    "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
    "Thymocyte", "Spleen",
    "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
    "A431", "HACAT", "HNSCC","Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
    "Vascular Smooth Muscle", "LHCNM2","Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
    "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
    "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron","Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
    "Trachea","Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
    "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
    "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]

category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}

In [23]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
import re

qn_matrix = np.array([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
       [11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
       [21, 22, 23, 24, 25, 26, 27, 28, 29, 30],
       [31, 32, 33, 34, 35, 36, 37, 38, 39, 40],
       [41, 42, 43, 44, 45, 46, 47, 48, 49, 50]])


gene_names = ["gene1", "gene2", "gene3", "gene4", "gene5"]

category_list  = ["Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", 
                  "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell"]

In [24]:
from IPython.display import display

df = pd.DataFrame(qn_matrix, index=gene_names)
df.columns = category_list
display(df)

,Heart,Atrium,Cardiac Muscle Fiber,Pericardium,Valve,Adipose,Adipocyte,Myofibroblast,Osteoblast,Stromal Cell
gene1,1,2,3,4,5,6,7,8,9,10
gene2,11,12,13,14,15,16,17,18,19,20
gene3,21,22,23,24,25,26,27,28,29,30
gene4,31,32,33,34,35,36,37,38,39,40
gene5,41,42,43,44,45,46,47,48,49,50


In [10]:
def get_gene_indexes(gene_names):
    gene_indexes = []
    new_gene_names = []
    for gene in gene_names:
        try:
            index = arch_gene_names.index(gene)
            gene_indexes.append(index)
            new_gene_names.append(gene)
        except ValueError:
            pass
    return new_gene_names, gene_indexes

gene_names, gene_indexes = get_gene_indexes(gene_names)

In [53]:
def get_geneset_category_means(matrix):
    
    # Calculate the column-wise average
    geneset_mean_row = np.mean(matrix, axis=0)

    return geneset_mean_row
    

In [57]:
# Initialize the final dictionary with empty lists for each category
# folder = "/home/lwu/ARCHS4/Tests/Check If Biosample Categorys Align With Genesets/"
# filename = folder + "h.c1.c2.all.v7.5.1.symbols.gmt"
filename = "/home/lwu/ARCHS4/Tests/top_10_genesets_per_category/test_cases/test_geneset.gmt"

final_category_means = {category: [] for categories in category_dict.values() for category in categories}

geneset_means_matrix = []
geneset_names = []
gene_names = []


with open(filename, 'r', ) as file:
    
    for line in file:
#         print(line)
        
        # Extract the geneset name and the corresponding genes 
        line = line.strip().split('\t')
        geneset_names.append(line[0])
        gene_names = line[2:]
#         print(geneset_names, ":", gene_names)
        
        # Match geneset genes to the ARCHS4 gene names list 
        # If gene not found, remove from the gene_names
        # Otherwise, return a list of the genes idsx within ARCHS4
        gene_names, gene_indexes = get_gene_indexes(gene_names)
#         print(gene_names, "\n", gene_indexes)
        
        extracted_matrix = qn_matrix[gene_indexes, :] 
#         print(extracted_matrix)
#          print("Extracted Matrix Shape:", extracted_matrix.shape)
        
        geneset_mean_row = get_geneset_category_means(extracted_matrix)
        geneset_means_matrix.append(geneset_mean_row)
        #print("Geneset Matrix Shape:", len(geneset_means_matrix), len(geneset_means_matrix[0]))
#         print()
#         print(geneset_means_matrix)

#print(geneset_means_matrix)
# print(geneset_names)
# View pandas df
print(geneset_names)
df = pd.DataFrame(geneset_means_matrix, columns=category_list, index=geneset_names)
# #print(df.shape)
# print(df)
#         print()
#        print("------------------------------------")
#        break
        
        
df.to_csv('/home/lwu/ARCHS4/Tests/top_10_genesets_per_category/test_geneset_means_matrix.csv', index=True)
display(df)



['HALLMARK_TNFA_SIGNALING_VIA_NFKB', 'HALLMARK_HYPOXIA', 'HALLMARK_CHOLESTEROL_HOMEOSTASIS']


/home/lwu/.conda/envs/example_env/lib/python3.8/site-packages/numpy/core/fromnumeric.py:3474: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/lwu/.conda/envs/example_env/lib/python3.8/site-packages/numpy/core/_methods.py:181: RuntimeWarning: invalid value encountered in true_divide
  ret = um.true_divide(


,Heart,Atrium,Cardiac Muscle Fiber,Pericardium,Valve,Adipose,Adipocyte,Myofibroblast,Osteoblast,Stromal Cell
HALLMARK_TNFA_SIGNALING_VIA_NFKB,18.5,19.5,20.5,21.5,22.5,23.5,24.5,25.5,26.5,27.5
HALLMARK_HYPOXIA,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0
HALLMARK_CHOLESTEROL_HOMEOSTASIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df

,Heart,Atrium,Alpha Cell,Skin,Lung,Testis
HALLMARK_TNFA_SIGNALING_VIA_NFKB,20.166667,20.5,23.5,21.5,25.5,24.5
HALLMARK_HYPOXIA,7.666667,8.0,11.0,9.0,13.0,12.0
HALLMARK_CHOLESTEROL_HOMEOSTASIS,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
arch_gene_names

['JUNB', 'CXCL2', 'ATF3', 'test', 'NFKBIA']

In [44]:
category_list

['Heart', 'Atrium', 'Alpha Cell', 'Skin', 'Lung', 'Testis']

In [47]:
geneset_names

['HALLMARK_TNFA_SIGNALING_VIA_NFKB']

In [48]:
df

,Heart,Atrium,Alpha Cell,Skin,Lung,Testis
HALLMARK_TNFA_SIGNALING_VIA_NFKB,20.166667,20.5,23.5,21.5,25.5,24.5
HALLMARK_HYPOXIA,7.666667,8.0,11.0,9.0,13.0,12.0
HALLMARK_CHOLESTEROL_HOMEOSTASIS,NaN,NaN,NaN,NaN,NaN,NaN
